# SO-ARM101 强化学习训练

在 Google Colab 上训练 SO-ARM101 机械臂

## 1. 检查 GPU

In [1]:
!nvidia-smi
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Sat Jul  4 02:19:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. 安装依赖

In [2]:
!pip install mujoco stable-baselines3 gymnasium tensorboard

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.7/232.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 28.2 MB/s eta 0:00:00


## 3. 上传项目文件

运行下面的代码，然后点击 "Choose Files" 上传你的 `so_arm101_rl_project` 文件夹中的所有文件。

**需要上传的文件：**
- `envs/` 文件夹（所有文件）
- `assets/` 文件夹（所有文件）
- `scripts/train_my_env.py`
- `requirements.txt`

In [3]:
import os
from google.colab import files

# 创建目录结构
os.makedirs('so_arm101_rl_project/envs', exist_ok=True)
os.makedirs('so_arm101_rl_project/assets', exist_ok=True)
os.makedirs('so_arm101_rl_project/scripts', exist_ok=True)
os.makedirs('so_arm101_rl_project/checkpoints', exist_ok=True)
os.makedirs('so_arm101_rl_project/logs', exist_ok=True)

print("目录结构创建完成！")
print("\n现在请上传文件：")
print("1. envs/ 文件夹中的所有 .py 文件")
print("2. assets/ 文件夹中的所有文件")
print("3. scripts/train_my_env.py")
print("\n上传完成后，运行下一个代码块。")

目录结构创建完成！

现在请上传文件：
1. envs/ 文件夹中的所有 .py 文件
2. assets/ 文件夹中的所有文件
3. scripts/train_my_env.py

上传完成后，运行下一个代码块。


In [ ]:
# 上传文件
uploaded = files.upload()

# 移动文件到正确目录
for filename in uploaded.keys():
    if 'envs' in filename:
        os.rename(filename, f'so_arm101_rl_project/envs/{os.path.basename(filename)}')
    elif 'assets' in filename or filename.endswith('.xml') or filename.endswith('.stl'):
        os.rename(filename, f'so_arm101_rl_project/assets/{os.path.basename(filename)}')
    elif 'train' in filename:
        os.rename(filename, f'so_arm101_rl_project/scripts/{os.path.basename(filename)}')
    else:
        os.rename(filename, f'so_arm101_rl_project/{os.path.basename(filename)}')

print("\n文件上传完成！")
print("\n目录结构：")
!find so_arm101_rl_project -type f | head -20

## 4. 检查文件结构

In [ ]:
# 检查文件是否存在
import os

required_files = [
    'so_arm101_rl_project/envs/__init__.py',
    'so_arm101_rl_project/envs/Mso_arm101_reach.py',
    'so_arm101_rl_project/assets/scene.xml',
    'so_arm101_rl_project/assets/so101_new_calib.xml',
    'so_arm101_rl_project/scripts/train_my_env.py'
]

print("检查必需文件：")
all_exist = True
for file in required_files:
    exists = os.path.exists(file)
    status = "✓" if exists else "✗"
    print(f"{status} {file}")
    if not exists:
        all_exist = False

if all_exist:
    print("\n✓ 所有文件都已就位！")
else:
    print("\n✗ 缺少文件，请重新上传！")

## 5. 修改并行环境数量

Colab 免费版通常有 2 个 CPU 核心，建议使用 2-4 个并行环境。

In [ ]:
# 修改 train_my_env.py 中的并行环境数量
import re

train_file = 'so_arm101_rl_project/scripts/train_my_env.py'

# 读取文件
with open(train_file, 'r', encoding='utf-8') as f:
    content = f.read()

# 修改并行环境数量为 4
content = re.sub(
    r'SubprocVecEnv\(\[make_env for _ in range\(\d+\)\]\)',
    'SubprocVecEnv([make_env for _ in range(4)])',
    content
)

# 修改训练步数为 1M
content = re.sub(
    r'total_timesteps=\d+',
    'total_timesteps=1000000',
    content
)

# 保存文件
with open(train_file, 'w', encoding='utf-8') as f:
    f.write(content)

print("✓ 配置更新完成！")
print("  - 并行环境数: 4")
print("  - 训练步数: 1,000,000")

## 6. 开始训练

In [ ]:
# 切换到项目目录并开始训练
import os
os.chdir('so_arm101_rl_project')

print("=" * 60)
print("开始训练 SO-ARM101")
print("=" * 60)
print("\n训练配置：")
print("  - 并行环境: 4")
print("  - 训练步数: 1,000,000")
print("  - 预计时间: 30-60 分钟")
print("\n" + "=" * 60 + "\n")

# 运行训练脚本
!python scripts/train_my_env.py

## 7. 查看训练结果

In [ ]:
# 检查训练结果
import os

print("\n训练结果：")

# 检查模型文件
model_files = [
    'checkpoints/best_model.zip',
    'ppo_my_env_reach.zip',
    'scripts/ppo_my_env_reach.zip'
]

for model_file in model_files:
    if os.path.exists(model_file):
        size = os.path.getsize(model_file) / 1024 / 1024  # MB
        print(f"✓ {model_file} ({size:.2f} MB)")

# 检查日志
if os.path.exists('logs'):
    log_files = []
    for root, dirs, files in os.walk('logs'):
        for file in files:
            log_files.append(os.path.join(root, file))
    print(f"\n✓ 生成了 {len(log_files)} 个日志文件")

print("\n" + "=" * 60)
print("训练完成！")
print("=" * 60)

## 8. 下载训练好的模型

In [ ]:
# 打包并下载训练结果
import os
from google.colab import files

# 打包模型和日志
!zip -r training_results.zip checkpoints/ logs/ ppo_my_env_reach.zip scripts/ppo_my_env_reach.zip 2>/dev/null || true

if os.path.exists('training_results.zip'):
    print("\n✓ 训练结果已打包: training_results.zip")
    print("\n下载文件...")
    files.download('training_results.zip')
else:
    print("\n✗ 未找到训练结果文件")
    print("\n尝试下载单个文件：")

    # 尝试下载单个文件
    for model_file in ['checkpoints/best_model.zip', 'ppo_my_env_reach.zip', 'scripts/ppo_my_env_reach.zip']:
        if os.path.exists(model_file):
            print(f"下载: {model_file}")
            files.download(model_file)

## 9. 可视化训练曲线（可选）

如果你想查看 TensorBoard 训练曲线，可以运行下面的代码。

In [ ]:
# 启动 TensorBoard
%load_ext tensorboard
%tensorboard --logdir logs